# FACUNDO - Best and worst boxers

This notebook contains Facundo's contribution only.

## Requirements
Run the shared notebook to create and populate `boxers.db`. Place that database in the same folder as this notebook. In Google Colab, upload `boxers.db` through the Files panel before running these cells.

## Ranking criteria
- `win_rate`: wins / total_fights * 100.
- `points`: 3 points per win and 1 per draw.

The function shows all tied boxers, excludes boxers with no fights, and opens the database in read-only mode.


In [1]:
from contextlib import closing
import sqlite3

import pandas as pd


In [2]:
def show_best_and_worst_boxer(metric: str = "win_rate") -> None:
  """
  Find and print the best and worst boxers from the database.

  Boxers with no fights are excluded. If several boxers share the
  highest or lowest score, all of them are shown.

  Args:
    metric (str, optional): "win_rate" calculates wins / total_fights
      as a percentage. "points" assigns 3 points per win and 1 per draw.
      Defaults to "win_rate".

  Returns:
    None
  """
  if metric not in ("win_rate", "points"):
    print('Error: Choose "win_rate" or "points".')
    return

  try:
    with closing(sqlite3.connect("file:boxers.db?mode=ro", uri=True)) as connection:
      boxers = pd.read_sql_query(
        "SELECT id, name, total_fights, wins, loses, draws FROM boxers",
        con=connection
      )

    if boxers.empty:
      print("There are no boxers in the database.")
      return

    boxers = boxers[boxers["total_fights"] > 0].copy()

    if boxers.empty:
      print("There are no boxers with fights to compare.")
      return

    if metric == "win_rate":
      boxers["score"] = boxers["wins"] / boxers["total_fights"] * 100
    else:
      boxers["score"] = boxers["wins"] * 3 + boxers["draws"]

    best_boxers = boxers[boxers["score"] == boxers["score"].max()]
    worst_boxers = boxers[boxers["score"] == boxers["score"].min()]

    print(f"Best boxer(s) by {metric}:")
    print(best_boxers.to_string(index=False, float_format="{:.2f}".format))
    print(f"\nWorst boxer(s) by {metric}:")
    print(worst_boxers.to_string(index=False, float_format="{:.2f}".format))

  except (sqlite3.Error, pd.errors.DatabaseError) as error:
    print("Error reading the database ->", error)


## Examples

Run the following cells to compare both ranking criteria.


In [3]:
show_best_and_worst_boxer()


Best boxer(s) by win_rate:
 id                 name  total_fights  wins  loses  draws  score
  7 Floyd Mayweather Jr.            50    50      0      0 100.00
  9      Oleksander Usyk            25    25      0      0 100.00

Worst boxer(s) by win_rate:
 id              name  total_fights  wins  loses  draws  score
  6 Evander Holyfield            57    44     10      2  77.19


In [4]:
show_best_and_worst_boxer("points")


Best boxer(s) by points:
 id               name  total_fights  wins  loses  draws  score
  3 Julio Cesar Chavez           115   107      6      2    323

Worst boxer(s) by points:
 id            name  total_fights  wins  loses  draws  score
  9 Oleksander Usyk            25    25      0      0     75
